# Exploração inicial — Hotel Booking Demand

Fase 0 do pipeline medalhão. Objetivo: entender o formato real dos 3 CSVs em `data/raw/` antes de definir os schemas explícitos da Bronze e as regras da Silver.

Pontos verificados:
1. Cabeçalhos, schema inferido e contagem de linhas
2. Valores distintos das colunas categóricas
3. Formato real de `reservation_status_date`
4. Ocorrências dos literais `"NULL"` e `"NA"` por coluna
5. Colunas das tabelas de referência
6. Pontos conhecidos: `children = "NA"`, reservas sem hóspedes, `adr` negativo, linhas duplicadas, países sem metadado
7. Sinais de risco de cancelamento conhecidos no momento da reserva

In [1]:
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

spark = (
    SparkSession.builder.master("local[*]")
    .appName("exploracao_inicial")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)
sorted(p.name for p in RAW.glob("*.csv"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 12:19:42 WARN Utils: Your hostname, lholiv-Vivobook-ASUSLaptop-X1504VA-X1504VA, resolves to a loopback address: 127.0.1.1; using 172.20.10.6 instead (on interface wlo1)
26/10/11 12:19:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/11 12:19:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.3


['country_metadata.csv', 'hotel_bookings.csv', 'hotel_metadata.csv']

## 1. Cabeçalhos, schema inferido e contagem de linhas

In [2]:
def read_csv(name: str, infer: bool = True, null_value: str | None = "NULL") -> DataFrame:
    reader = spark.read.option("header", True).option("sep", ",").option("inferSchema", infer)
    if null_value is not None:
        reader = reader.option("nullValue", null_value)
    return reader.csv(str(RAW / name))

bookings = read_csv("hotel_bookings.csv")
countries = read_csv("country_metadata.csv")
hotels = read_csv("hotel_metadata.csv")

for name, df in [("hotel_bookings", bookings), ("country_metadata", countries), ("hotel_metadata", hotels)]:
    print(f"{name}: {df.count():,} linhas x {len(df.columns)} colunas")

hotel_bookings: 119,390 linhas x 32 colunas


country_metadata: 252 linhas x 3 colunas


hotel_metadata: 2 linhas x 5 colunas


In [3]:
bookings.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

Observações sobre o schema inferido (com `nullValue="NULL"`):
- `children` é inferido como **string** por causa dos valores `"NA"` (ver seção 4).
- `agent` e `company` viram inteiros porque `"NULL"` foi tratado como nulo.
- `reservation_status_date` é inferido como `date`, o que indica formato ISO (confirmado na seção 3).

In [4]:
bookings.limit(5).toPandas().T

,0,1,2,3,4
hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel
is_canceled,0,0,0,0,0
lead_time,342,737,7,13,14
arrival_date_year,2015,2015,2015,2015,2015
arrival_date_month,July,July,July,July,July
arrival_date_week_number,27,27,27,27,27
arrival_date_day_of_month,1,1,1,1,1
stays_in_weekend_nights,0,0,0,0,0
stays_in_week_nights,0,0,1,1,2
adults,2,2,1,1,2


## 2. Valores distintos das colunas categóricas

In [5]:
categorical = [
    "hotel", "arrival_date_month", "meal", "market_segment", "distribution_channel",
    "deposit_type", "customer_type", "reservation_status",
]
for col in categorical:
    print(f"--- {col}")
    bookings.groupBy(col).count().orderBy(F.desc("count")).show(truncate=False)

--- hotel


+------------+-----+
|hotel       |count|
+------------+-----+
|City Hotel  |79330|
|Resort Hotel|40060|
+------------+-----+

--- arrival_date_month


+------------------+-----+
|arrival_date_month|count|
+------------------+-----+
|August            |13877|
|July              |12661|
|May               |11791|
|October           |11160|
|April             |11089|
|June              |10939|
|September         |10508|
|March             |9794 |
|February          |8068 |
|November          |6794 |
|December          |6780 |
|January           |5929 |
+------------------+-----+

--- meal


+---------+-----+
|meal     |count|
+---------+-----+
|BB       |92310|
|HB       |14463|
|SC       |10650|
|Undefined|1169 |
|FB       |798  |
+---------+-----+

--- market_segment


+--------------+-----+
|market_segment|count|
+--------------+-----+
|Online TA     |56477|
|Offline TA/TO |24219|
|Groups        |19811|
|Direct        |12606|
|Corporate     |5295 |
|Complementary |743  |
|Aviation      |237  |
|Undefined     |2    |
+--------------+-----+

--- distribution_channel


+--------------------+-----+
|distribution_channel|count|
+--------------------+-----+
|TA/TO               |97870|
|Direct              |14645|
|Corporate           |6677 |
|GDS                 |193  |
|Undefined           |5    |
+--------------------+-----+

--- deposit_type


+------------+------+
|deposit_type|count |
+------------+------+
|No Deposit  |104641|
|Non Refund  |14587 |
|Refundable  |162   |
+------------+------+

--- customer_type


+---------------+-----+
|customer_type  |count|
+---------------+-----+
|Transient      |89613|
|Transient-Party|25124|
|Contract       |4076 |
|Group          |577  |
+---------------+-----+

--- reservation_status


+------------------+-----+
|reservation_status|count|
+------------------+-----+
|Check-Out         |75166|
|Canceled          |43017|
|No-Show           |1207 |
+------------------+-----+



- `arrival_date_month` traz os 12 meses por extenso em inglês, sem variações de caixa: o mapeamento para 1–12 pode ser um dicionário fixo.
- `market_segment` e `distribution_channel` têm a categoria `Undefined` (poucas linhas); `meal` tem `Undefined` e `SC` (equivalentes segundo o artigo original). Ficam como vieram.
- `reservation_status` tem exatamente `Check-Out`, `Canceled` e `No-Show`.

In [6]:
# is_canceled x reservation_status: No-Show conta como cancelado?
bookings.groupBy("reservation_status", "is_canceled").count().orderBy("reservation_status").show()

+------------------+-----------+-----+
|reservation_status|is_canceled|count|
+------------------+-----------+-----+
|          Canceled|          1|43017|
|         Check-Out|          0|75166|
|           No-Show|          1| 1207|
+------------------+-----------+-----+



`No-Show` aparece sempre com `is_canceled = 1`. Logo, `revenue = adr * total_nights` apenas quando `is_canceled = 0` exclui também os No-Shows.

## 3. Formato real de `reservation_status_date`

In [7]:
bookings_str = read_csv("hotel_bookings.csv", infer=False, null_value=None)  # tudo string, sem nullValue

status_date = bookings_str.select("reservation_status_date")
status_date.show(5)
(
    status_date
    .withColumn(
        "pattern",
        F.when(F.col("reservation_status_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
        .when(F.col("reservation_status_date").rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"), "M/d/yyyy")
        .otherwise("other"),
    )
    .groupBy("pattern").count()
    .show()
)
bookings.select(
    F.min("reservation_status_date").alias("min_date"),
    F.max("reservation_status_date").alias("max_date"),
).show()

+-----------------------+
|reservation_status_date|
+-----------------------+
|             2015-07-01|
|             2015-07-01|
|             2015-07-02|
|             2015-07-02|
|             2015-07-03|
+-----------------------+
only showing top 5 rows


+----------+------+
|   pattern| count|
+----------+------+
|yyyy-MM-dd|119390|
+----------+------+



+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2014-10-17|2017-09-14|
+----------+----------+



## 4. Ocorrências dos literais `"NULL"` e `"NA"` por coluna

In [8]:
literal_counts = bookings_str.select(
    *[F.sum((F.col(c) == "NULL").cast("int")).alias(c) for c in bookings_str.columns]
).toPandas().T.rename(columns={0: "NULL"})
literal_counts["NA"] = bookings_str.select(
    *[F.sum((F.col(c) == "NA").cast("int")).alias(c) for c in bookings_str.columns]
).toPandas().T[0]
literal_counts["empty"] = bookings_str.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in bookings_str.columns]
).toPandas().T[0]
literal_counts[(literal_counts > 0).any(axis=1)]

,NULL,NA,empty
children,0,4,0
country,488,0,0
agent,16340,0,0
company,112593,0,0


- `"NULL"` aparece em `country`, `agent` e `company`: coberto pelo `nullValue="NULL"`.
- `"NA"` aparece apenas em `children` e **não** é coberto pelo `nullValue`. Decisão: na Bronze, `children` é lido como **string** (fiel à fonte); o cast para inteiro (`"NA"` → nulo → 0) acontece na Silver.
- Não há campos vazios.

In [9]:
bookings_str.groupBy("children").count().orderBy("children").show()

+--------+------+
|children| count|
+--------+------+
|       0|110796|
|       1|  4861|
|      10|     1|
|       2|  3652|
|       3|    76|
|      NA|     4|
+--------+------+



## 5. Tabelas de referência

In [10]:
countries.printSchema()
hotels.printSchema()
hotels.show(truncate=False)
countries.groupBy("continent").count().orderBy(F.desc("count")).show(truncate=False)

root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- continent: string (nullable = true)

root
 |-- hotel_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- opened_year: integer (nullable = true)

+------------+-------+------------+-----------+-----------+
|hotel_name  |city   |country_code|star_rating|opened_year|
+------------+-------+------------+-----------+-----------+
|Resort Hotel|Algarve|PRT         |4          |2010       |
|City Hotel  |Lisbon |PRT         |4          |2008       |
+------------+-------+------------+-----------+-----------+



+-------------+-----+
|continent    |count|
+-------------+-----+
|Africa       |59   |
|Europe       |56   |
|Asia         |49   |
|North America|40   |
|Oceania      |26   |
|South America|14   |
|Antarctica   |5    |
|Unknown      |3    |
+-------------+-----+



In [11]:
# Chaves únicas? Duplicidade na chave da dimensão duplicaria reservas no LEFT JOIN.
for name, df, key in [("country_metadata", countries, "country_code"), ("hotel_metadata", hotels, "hotel_name")]:
    total, distinct = df.count(), df.select(key).distinct().count()
    print(f"{name}.{key}: {total} linhas, {distinct} chaves distintas")

# Espaços extras nas chaves?
print("country_code com espaços:", countries.filter(F.col("country_code") != F.trim("country_code")).count())
print("hotel_name com espaços:", hotels.filter(F.col("hotel_name") != F.trim("hotel_name")).count())

country_metadata.country_code: 252 linhas, 252 chaves distintas


hotel_metadata.hotel_name: 2 linhas, 2 chaves distintas


country_code com espaços: 0


hotel_name com espaços: 0


## 6. Pontos conhecidos do dataset

### 6.1 Reservas sem hóspedes e `adr` negativo

In [12]:
# try_cast: com ANSI ligado (padrão no Spark 4), cast("int") de "NA" falha
children_int = F.coalesce(F.col("children").try_cast("int"), F.lit(0))
guests = F.col("adults") + children_int + F.col("babies")

print("adults + children + babies = 0:", bookings.filter(guests == 0).count())
print("adr < 0:", bookings.filter(F.col("adr") < 0).count())
print("adr = 0:", bookings.filter(F.col("adr") == 0).count())
print("ambos (sem hóspedes e adr < 0):", bookings.filter((guests == 0) & (F.col("adr") < 0)).count())

bookings.filter(F.col("adr") < 0).select(
    "hotel", "is_canceled", "arrival_date_year", "arrival_date_month", "adults", "children", "babies",
    "stays_in_weekend_nights", "stays_in_week_nights", "adr", "reservation_status",
).show()
bookings.select(F.expr("percentile_approx(adr, array(0.5, 0.99, 0.999))").alias("adr_p50_p99_p999"), F.max("adr").alias("adr_max")).show(truncate=False)

adults + children + babies = 0: 180


adr < 0: 1


adr = 0: 1959


ambos (sem hóspedes e adr < 0): 0


+------------+-----------+-----------------+------------------+------+--------+------+-----------------------+--------------------+-----+------------------+
|       hotel|is_canceled|arrival_date_year|arrival_date_month|adults|children|babies|stays_in_weekend_nights|stays_in_week_nights|  adr|reservation_status|
+------------+-----------+-----------------+------------------+------+--------+------+-----------------------+--------------------+-----+------------------+
|Resort Hotel|          0|             2017|             March|     2|       0|     0|                      4|                   6|-6.38|         Check-Out|
+------------+-----------+-----------------+------------------+------+--------+------+-----------------------+--------------------+-----+------------------+



+---------------------+-------+
|adr_p50_p99_p999     |adr_max|
+---------------------+-------+
|[94.5, 251.73, 324.0]|5400.0 |
+---------------------+-------+



#### Cauda superior de `adr`

O p99.9 é 324 e o máximo é 5.400. Antes de decidir o que fazer com o máximo, vale ver se existe uma cauda contínua até ele ou um salto isolado.

In [13]:
print("adr > 500:", bookings.filter(F.col("adr") > 500).count())
print("adr > 1000:", bookings.filter(F.col("adr") > 1000).count())

# Valores acima de 400, em ordem: mostra se a cauda é contínua ou tem um salto
(
    bookings.filter(F.col("adr") > 400)
    .select("hotel", "is_canceled", "market_segment", "arrival_date_year", "arrival_date_month", "adults", "children", "adr")
    .orderBy("adr")
    .show(truncate=False)
)

adr > 500: 3


adr > 1000: 1


+------------+-----------+--------------+-----------------+------------------+------+--------+------+
|hotel       |is_canceled|market_segment|arrival_date_year|arrival_date_month|adults|children|adr   |
+------------+-----------+--------------+-----------------+------------------+------+--------+------+
|Resort Hotel|0          |Online TA     |2017             |August            |3     |1       |402.0 |
|Resort Hotel|0          |Direct        |2017             |August            |2     |2       |426.25|
|Resort Hotel|1          |Direct        |2017             |August            |2     |2       |437.0 |
|Resort Hotel|1          |Online TA     |2017             |August            |2     |0       |450.0 |
|City Hotel  |0          |Direct        |2016             |December          |2     |2       |451.5 |
|Resort Hotel|0          |Corporate     |2015             |July              |2     |0       |508.0 |
|City Hotel  |0          |Offline TA/TO |2017             |May               |1   

In [14]:
# O outlier: está cancelado? Isso define se ele afeta revenue e avg_adr
(
    bookings.filter(F.col("adr") > 1000)
    .select(
        "hotel", "is_canceled", "reservation_status", "deposit_type", "market_segment",
        "arrival_date_year", "arrival_date_month", "stays_in_weekend_nights", "stays_in_week_nights",
        "adults", "children", "adr",
    )
    .show(truncate=False)
)

+----------+-----------+------------------+------------+--------------+-----------------+------------------+-----------------------+--------------------+------+--------+------+
|hotel     |is_canceled|reservation_status|deposit_type|market_segment|arrival_date_year|arrival_date_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|adr   |
+----------+-----------+------------------+------------+--------------+-----------------+------------------+-----------------------+--------------------+------+--------+------+
|City Hotel|1          |Canceled          |Non Refund  |Offline TA/TO |2016             |March             |0                      |1                   |2     |0       |5400.0|
+----------+-----------+------------------+------------+--------------+-----------------+------------------+-----------------------+--------------------+------+--------+------+



Os valores crescem sem saltos até 510 e depois pulam para 5.400, em uma única reserva: provável erro de digitação. O registro é **mantido**, porque o enunciado só considera inválido `adr < 0`, e mantê-lo é seguro porque a reserva está **cancelada**: `revenue = 0` e ela fica fora do `avg_adr`. Na Silver, a regra de alerta `adr_within_expected_range` (`adr <= 1000`) sinaliza casos assim em cada execução; o limite de 1.000 separa o outlier sem marcar diárias altas plausíveis.

### 6.2 Linhas inteiramente duplicadas

In [15]:
total = bookings.count()
distinct_rows = bookings.distinct().count()
print(f"total: {total:,} | distintas: {distinct_rows:,} | excedentes duplicadas: {total - distinct_rows:,} ({(total - distinct_rows) / total:.1%})")

(
    bookings.groupBy(*bookings.columns).count()
    .filter(F.col("count") > 1)
    .groupBy(F.col("count").alias("copies"))
    .agg(F.count("*").alias("distinct_rows"))
    .orderBy("copies")
    .show(10)
)

total: 119,390 | distintas: 87,396 | excedentes duplicadas: 31,994 (26.8%)


+------+-------------+
|copies|distinct_rows|
+------+-------------+
|     2|         5034|
|     3|          969|
|     4|          497|
|     5|          275|
|     6|          171|
|     7|          115|
|     8|          103|
|     9|           93|
|    10|          123|
|    11|           60|
+------+-------------+
only showing top 10 rows


Não existe ID de reserva no dataset. Sem chave de negócio, não dá para distinguir uma duplicata técnica de reservas idênticas legítimas (ex.: grupo que reserva vários quartos iguais no mesmo dia). **Decisão: não deduplicar**; a contagem fica documentada no README.

### 6.3 Países sem correspondência em `country_metadata`

In [16]:
country_keys = countries.select(F.col("country_code").alias("country"), F.lit(1).alias("_has_meta"))
joined = bookings.join(country_keys, on="country", how="left")

print("bookings com country nulo (\"NULL\"):", bookings.filter(F.col("country").isNull()).count())
unmatched = joined.filter(F.col("country").isNotNull() & F.col("_has_meta").isNull())
print("bookings com country preenchido e sem metadado:", unmatched.count())
print("códigos distintos sem metadado:", unmatched.select("country").distinct().count())
unmatched.groupBy("country").count().orderBy(F.desc("count")).show(50)

bookings com country nulo ("NULL"): 488


bookings com country preenchido e sem metadado: 0


códigos distintos sem metadado: 0


+-------+-----+
|country|count|
+-------+-----+
+-------+-----+



In [17]:
print("países distintos nas reservas:", bookings.select("country").where(F.col("country").isNotNull()).distinct().count())
print("países em country_metadata:", countries.count())
print("hotéis sem metadado:", bookings.join(hotels, bookings.hotel == hotels.hotel_name, "left_anti").count())

países distintos nas reservas: 177
países em country_metadata: 252


hotéis sem metadado: 0


In [18]:
# Códigos fora do padrão ISO-3 nas reservas e países com continente "Unknown"
bookings.filter(F.length("country") != 3).groupBy("country").count().show()
countries.filter(F.col("country_name") == "China").show()
bookings.filter(F.col("country").isin("CN", "CHN")).groupBy("country").count().show()
unknown = countries.filter(F.col("continent") == "Unknown")
unknown.join(
    bookings.groupBy(F.col("country").alias("country_code")).count(), "country_code", "left"
).show(truncate=False)

+-------+-----+
|country|count|
+-------+-----+
|     CN| 1279|
+-------+-----+

+------------+------------+---------+
|country_code|country_name|continent|
+------------+------------+---------+
|         CHN|       China|     Asia|
|          CN|       China|     Asia|
+------------+------------+---------+



+-------+-----+
|country|count|
+-------+-----+
|     CN| 1279|
|    CHN|  999|
+-------+-----+



+------------+------------------------------+---------+-----+
|country_code|country_name                  |continent|count|
+------------+------------------------------+---------+-----+
|BES         |Bonaire                       |Unknown  |NULL |
|IOT         |British Indian Ocean Territory|Unknown  |NULL |
|SJM         |Svalbard and Jan Mayen        |Unknown  |NULL |
+------------+------------------------------+---------+-----+



`country_metadata` tem 252 países (o enunciado menciona ~50), com chave única, e cobre todos os 177 códigos presentes nas reservas. A cobertura do metadado é total; o que fica sem `country_name` são apenas as 488 reservas com `country = "NULL"`, que viram `"UNK"` na Silver.

Ponto de atenção: as reservas usam `CN` (1.279) e `CHN` (999) para a China, e o `country_metadata` mapeia **os dois** códigos para `China`. Sem tratamento, a China apareceria em duas linhas na visão Gold de Top países por `(country, country_name, continent)`; a análise abaixo mostra que o caso não é isolado. Os 3 países com continente `Unknown` não têm reservas.

#### Nomes de país repetidos em `country_metadata`

A chave `country_code` é única, mas `country_name` não. É preciso separar dois casos: aliases do mesmo país, que devem ser unificados, e países diferentes com nome genérico, que nunca podem ser agrupados pelo nome.

In [19]:
repeated_names = (
    countries.groupBy("country_name")
    .agg(F.count("*").alias("codes"))
    .filter(F.col("codes") > 1)
    .select("country_name")
)
bookings_per_code = bookings.groupBy(F.col("country").alias("country_code")).agg(F.count("*").alias("bookings"))

(
    countries.join(repeated_names, "country_name")
    .join(bookings_per_code, "country_code", "left")
    .fillna({"bookings": 0})
    .select("country_name", "country_code", "continent", "bookings")
    .orderBy("country_name", "country_code")
    .show(truncate=False)
)

+--------------+------------+-------------+--------+
|country_name  |country_code|continent    |bookings|
+--------------+------------+-------------+--------+
|Angola        |AGO         |Africa       |362     |
|Angola        |ANG         |Africa       |0       |
|China         |CHN         |Asia         |999     |
|China         |CN          |Asia         |1279    |
|Congo         |COD         |Africa       |0       |
|Congo         |COG         |Africa       |0       |
|Korea         |KOR         |Asia         |133     |
|Korea         |PRK         |Asia         |0       |
|Timor-Leste   |TLS         |Asia         |0       |
|Timor-Leste   |TMP         |Asia         |3       |
|Virgin Islands|VGB         |North America|1       |
|Virgin Islands|VIR         |North America|0       |
+--------------+------------+-------------+--------+



- **Aliases do mesmo país:** `ANG` → `AGO`, `CN` → `CHN` (ISO-2 em uma coluna ISO-3) e `TMP` → `TLS` (ISO-3 obsoleto). Na Silver, esses códigos são convertidos para o ISO-3 atual antes do join, e o original fica em `country_raw`.
- **Países diferentes com nome genérico:** Korea (`KOR` = Coreia do Sul, `PRK` = Coreia do Norte), Congo (`COD`, `COG`) e Virgin Islands (`VGB` britânicas, `VIR` americanas). Hoje só um lado de cada par tem reservas, mas agrupar pelo nome juntaria países distintos se isso mudar. Por isso o join e os agrupamentos usam sempre o código.
- **Ressalva:** no dataset original, `CN` é ambíguo. A decisão segue o metadado fornecido, que o mapeia para China.

## 7. Sinais de risco de cancelamento

Pergunta do enunciado: como detectar reservas com risco alto de cancelamento para acionar campanhas de retenção? Aqui avaliamos sinais **conhecidos no momento da reserva**: histórico do hóspede (`previous_cancellations`, `previous_bookings_not_canceled`), antecedência (`lead_time`) e tipo de depósito. `reservation_status` e `reservation_status_date` ficam de fora, porque registram o desfecho da reserva (vazamento de informação).

As faixas são as mesmas da Silver (`add_cancellation_risk_features`), para que os números sejam comparáveis com a visão `cancellation_risk_profile`.

In [20]:
lead_time = F.col("lead_time")


def count_bucket(col: str):
    """Faixa 0 / 1 / 2+ para contadores de histórico."""
    return F.when(F.col(col) == 0, "0").when(F.col(col) == 1, "1").otherwise("2+")


risk = bookings.withColumns(
    {
        "lead_time_bucket": (
            F.when(lead_time <= 7, "000-007")
            .when(lead_time <= 30, "008-030")
            .when(lead_time <= 90, "031-090")
            .when(lead_time <= 180, "091-180")
            .when(lead_time <= 365, "181-365")
            .otherwise("366+")
        ),
        "previous_cancellations_bucket": count_bucket("previous_cancellations"),
        "previous_stays_bucket": count_bucket("previous_bookings_not_canceled"),
    }
)


def cancellation_rate_by(df: DataFrame, *cols: str) -> DataFrame:
    """Reservas e taxa de cancelamento por combinação de `cols`."""
    return (
        df.groupBy(*cols)
        .agg(F.count("*").alias("bookings"), F.round(F.avg("is_canceled"), 3).alias("cancellation_rate"))
        .orderBy(*cols)
    )


for col in ["previous_cancellations_bucket", "previous_stays_bucket", "lead_time_bucket", "deposit_type", "is_repeated_guest"]:
    cancellation_rate_by(risk, col).show(truncate=False)

+-----------------------------+--------+-----------------+
|previous_cancellations_bucket|bookings|cancellation_rate|
+-----------------------------+--------+-----------------+
|0                            |112906  |0.339            |
|1                            |6051    |0.944            |
|2+                           |433     |0.527            |
+-----------------------------+--------+-----------------+



+---------------------+--------+-----------------+
|previous_stays_bucket|bookings|cancellation_rate|
+---------------------+--------+-----------------+
|0                    |115770  |0.38             |
|1                    |1542    |0.051            |
|2+                   |2078    |0.058            |
+---------------------+--------+-----------------+



+----------------+--------+-----------------+
|lead_time_bucket|bookings|cancellation_rate|
+----------------+--------+-----------------+
|000-007         |19746   |0.096            |
|008-030         |18960   |0.279            |
|031-090         |29553   |0.377            |
|091-180         |26439   |0.447            |
|181-365         |21544   |0.555            |
|366+            |3148    |0.677            |
+----------------+--------+-----------------+



+------------+--------+-----------------+
|deposit_type|bookings|cancellation_rate|
+------------+--------+-----------------+
|No Deposit  |104641  |0.284            |
|Non Refund  |14587   |0.994            |
|Refundable  |162     |0.222            |
+------------+--------+-----------------+



+-----------------+--------+-----------------+
|is_repeated_guest|bookings|cancellation_rate|
+-----------------+--------+-----------------+
|0                |115580  |0.378            |
|1                |3810    |0.145            |
+-----------------+--------+-----------------+



Todos os sinais discriminam bem, mas `previous_cancellations` não é monotônico: 94% de cancelamento com 1 cancelamento anterior e 53% com 2+. Isso sugere que o efeito não é só comportamento individual. A próxima célula verifica se ele se mistura com reservas de grupo e agência.

In [21]:
with_previous = risk.filter(F.col("previous_cancellations") > 0)
total_with_previous = with_previous.count()

# Em que depósito e segmento estão as reservas com cancelamento anterior?
cancellation_rate_by(with_previous, "deposit_type", "market_segment").orderBy(F.desc("bookings")).show(10, truncate=False)

group_blocks = with_previous.filter(
    (F.col("deposit_type") == "Non Refund") & F.col("market_segment").isin("Groups", "Offline TA/TO")
).count()
not_repeated = with_previous.filter(F.col("is_repeated_guest") == 0).count()
print(f"com cancelamento anterior: {total_with_previous:,}")
print(f"  Non Refund em Groups ou Offline TA/TO: {group_blocks:,} ({group_blocks / total_with_previous:.1%})")
print(f"  is_repeated_guest = 0: {not_repeated:,} ({not_repeated / total_with_previous:.1%})")

+------------+--------------+--------+-----------------+
|deposit_type|market_segment|bookings|cancellation_rate|
+------------+--------------+--------+-----------------+
|Non Refund  |Groups        |2375    |1.0              |
|Non Refund  |Offline TA/TO |1224    |1.0              |
|No Deposit  |Groups        |973     |0.999            |
|No Deposit  |Online TA     |618     |0.939            |
|No Deposit  |Corporate     |530     |0.274            |
|No Deposit  |Offline TA/TO |455     |0.96             |
|No Deposit  |Direct        |187     |0.743            |
|No Deposit  |Complementary |66      |0.227            |
|Non Refund  |Corporate     |47      |1.0              |
|No Deposit  |Aviation      |9       |0.889            |
+------------+--------------+--------+-----------------+



com cancelamento anterior: 6,484
  Non Refund em Groups ou Offline TA/TO: 3,599 (55.5%)
  is_repeated_guest = 0: 5,557 (85.7%)


In [22]:
# A antecedência discrimina mesmo sem o efeito do Non Refund?
cancellation_rate_by(risk.filter(F.col("deposit_type") == "No Deposit"), "lead_time_bucket").show()

+----------------+--------+-----------------+
|lead_time_bucket|bookings|cancellation_rate|
+----------------+--------+-----------------+
|         000-007|   19619|            0.092|
|         008-030|   18438|             0.26|
|         031-090|   27139|            0.322|
|         091-180|   22260|            0.349|
|         181-365|   15905|            0.398|
|            366+|    1280|            0.206|
+----------------+--------+-----------------+



- **Os sinais se misturam:** 55,5% das reservas com cancelamento anterior são `Non Refund` em `Groups` ou `Offline TA/TO`, todas canceladas. O mais provável é que sejam lotes de agência cancelados em bloco. Por isso a visão `cancellation_risk_profile` cruza o histórico com `deposit_type` e `market_segment`, em vez de analisar o histórico isoladamente.
- **O histórico é do perfil no sistema de reservas, não da pessoa:** 85,7% dessas reservas têm `is_repeated_guest = 0`, e não há ID de hóspede para validar o dado.
- **Estadia anterior sem cancelamento protege:** a taxa cai de 38% para cerca de 5%.
- **A antecedência é um sinal forte, mas só em parte independente do depósito:** no total, a taxa sobe de 9,6% (até 7 dias) para 67,7% (mais de 1 ano). Dentro de `No Deposit`, ela continua crescendo até 1 ano (9,2% → 39,8%), o que confirma `lead_time` como sinal próprio, mas cai para 20,6% acima de 366 dias. O pico de 67,7% nessa faixa vem do `Non Refund`, concentrado em reservas de grupo feitas com muita antecedência. Mais um motivo para a visão cruzar antecedência e depósito.
- **`Non Refund` com 99,4% de cancelamento** contraria a intuição de negócio e é uma anomalia conhecida do dataset. Fica registrada, não corrigida.
- A visão Gold identifica **perfis** de risco. Atribuir um score a cada reserva exigiria um modelo preditivo, o que fica como próximo passo.

## Resumo dos achados

O resumo consolidado (com os números acima) está no README, seção "Achados de qualidade dos dados". Impacto direto na Fase 1:

- **Schema explícito da Bronze:** `children` como string; `reservation_status_date` como string na Bronze, convertido com `to_date(..., "yyyy-MM-dd")` na Silver; demais colunas numéricas como `int`/`double` (seguro com `nullValue="NULL"`).
- **Mês:** dicionário fixo `January..December → 1..12`.
- **`revenue`:** `is_canceled = 0`; No-Show já está em `is_canceled = 1`.
- **Duplicatas:** mantidas.
- **Países:** nulos viram `"UNK"`; códigos sem metadado ficam com `country_name`/`continent` nulos e são contados pela regra de alerta `has_country_metadata`.
- **`adr` 5.400:** um único registro, provável erro de digitação (cauda contínua até 510). Mantido, porque está cancelado e não afeta receita; monitorado pela regra de alerta `adr <= 1000`.
- **Códigos de país:** aliases `CN`/`TMP`/`ANG` convertidos para ISO-3 na Silver; nomes genéricos (Korea, Congo, Virgin Islands) levam a usar sempre o código no join e nos agrupamentos.
- **Risco de cancelamento:** histórico, antecedência e depósito discriminam bem, mas o histórico se mistura com reservas de grupo e agência `Non Refund`. A visão `cancellation_risk_profile` cruza esses sinais.

In [23]:
spark.stop()